# Ontologies (OCDI) at WeAreDevelopers

During this talk Konrad Bujak is showing his research into ontologies-based company AI workspaces. We are showing connecting 3 various stores, each containing semantic layers (datasources know how to query itself)

- Ontologies store: neo4j
- Embeddings: Cohere Embedd v4
- Vector Store: Qdrant (self-hosted)
- LLMs: OpenAI (Azure)

Datasources:
- sqlite
- vector store

Planned datasources:
- Snowflake
- MS SQL
- S3
- MongoDB

## Flow

1. inspect the curated corpus
2. run one live metadata extraction
3. build SQLite datastores for `vine_diseases` and `wine_diseases`
4. show the optional Qdrant path for `wine_making`
5. create the aggregated Neo4j graph
6. run manual Cypher to see what information is where
7. run the LangGraph workflow over the three live questions



In [1]:
import asyncio
import json
import os
import re
import runpy
import sqlite3
from collections import Counter, defaultdict
from pathlib import Path
from typing import Any, Literal, TypedDict

import cohere
import frontmatter
import pandas as pd
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langgraph.graph import END, START, StateGraph
from neo4j import GraphDatabase, RoutingControl
from openai import AsyncOpenAI
from pydantic import BaseModel, ConfigDict, Field, field_validator
from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider
from qdrant_client import QdrantClient, models

load_dotenv(dotenv_path=Path('.env'), override=False)

True

In [2]:
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / 'data' / 'winde_demo'
PROCESSED_DIR = PROJECT_ROOT / 'processed' / 'winde_demo'
JSONL_PATH = PROCESSED_DIR / 'metadata_extractions.jsonl'
CHUNK_JSONL_PATH = PROCESSED_DIR / 'wine_making_chunks.jsonl'
SQLITE_DIR = PROCESSED_DIR / 'sqlite'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
SQLITE_DIR.mkdir(parents=True, exist_ok=True)

AZURE_OPENAI_DEPLOYMENT = os.getenv('AZURE_OPENAI_DEPLOYMENT')
COHERE_EMBED_MODEL = os.getenv('COHERE_EMBED_MODEL', 'embed-v4.0')
DEMO_ID = os.getenv('WINE_DEMO_ID', 'neo4j-booth-wine-demo')


In [3]:
NEO4J_USERNAME = os.getenv('NEO4J_USERNAME', '<neo4j-username>')
NEO4J_PASSWORD = os.getenv('NEO4J_PASSWORD', '<password>')
NEO4J_URI = os.getenv('NEO4J_URI') or (
    f'neo4j+s://{NEO4J_USERNAME}.databases.neo4j.io')

In [4]:
REPOSITORY_ORDER = ['vine_diseases', 'wine_diseases', 'wine_making']
STORE_DESCRIPTIONS = {
    'vine_diseases': 'Vineyard disease pressure, cultivar susceptibility, and region-level growing risk.',
    'wine_diseases': 'How disease or botrytised fruit changes must and wine quality.',
    'wine_making': 'Cellar operations, clarification, filtration, enzymes, and process handling.',
}
TERM_FIELDS = [
    ('regions', 'Region'),
    ('varieties', 'Variety'),
    ('diseases', 'Disease'),
    ('winemaking_steps', 'Process'),
    ('quality_effects', 'QualityEffect'),
    ('keywords', 'Keyword'),
]

In [5]:
QDRANT_URL = os.getenv('QDRANT_URL', 'http://localhost:6333')
QDRANT_COLLECTION = os.getenv('WINE_DEMO_QDRANT_COLLECTION', 'wine_making_chunks')

## Inspect The Curated Corpus
The corpus already contains long topical briefs with `metadata:` frontmatter.



In [6]:
CORPUS_PATHS = sorted(DATA_ROOT.rglob('*.md'))
if not CORPUS_PATHS:
    raise RuntimeError(f'No markdown files found under {DATA_ROOT}')

print(f'Total markdown files: {len(CORPUS_PATHS)}')
pd.Series([path.parent.name for path in CORPUS_PATHS]).value_counts().rename_axis('repository').to_frame('documents')

Total markdown files: 16


,documents
repository,
vine_diseases,6
wine_diseases,5
wine_making,5


In [7]:
sample_path = CORPUS_PATHS[0]
sample_post = frontmatter.load(sample_path)

print(sample_path.relative_to(PROJECT_ROOT))
print(sample_post.metadata)
print()
print(sample_post.content[:200])



data/winde_demo/vine_diseases/france_gray_rot_pressure.md
{'title': 'Burgundy brief: why French vineyard questions about ripe grapes often become gray-rot questions', 'repository': 'vine_diseases', 'country': 'France', 'region': 'Burgundy', 'source': 'INRAE Ephytia', 'source_urls': ['https://ephytia.inrae.fr/en/C/6089/Grapevine-Botrytis-cinerea-Gray-mold', 'https://ephytia.inrae.fr/en/C/6979/Grapevine-Main-symptoms', 'https://hal.inrae.fr/hal-02735706/document'], 'metadata': {'summary': 'INRAE treats Botrytis cinerea as a major problem of ripe grapes in several French regions, especially Burgundy, Beaujolais, and the Loire Valley. This brief uses Burgundy as the anchor region for explaining how humid vineyard conditions, bunch infection, and pre-harvest decisions make gray rot a natural routing target for the vineyard-disease repository.', 'regions': ['France', 'Burgundy', 'Beaujolais', 'Loire Valley', 'Bordeaux'], 'varieties': ['Pinot Noir'], 'diseases': ['gray rot', 'Botrytis cinerea

## Models
`pydantic-ai` is used for extraction, triage, SQL translation, Qdrant query planning, and final synthesis.



In [8]:
def normalize_text(text: str) -> str:
    return re.sub(r'\s+', ' ', text or '').strip().lower()


def clean_list(value: Any) -> list[str]:
    if value is None:
        return []
    if isinstance(value, str):
        values = [value]
    else:
        values = list(value)

    cleaned: list[str] = []
    seen: set[str] = set()
    for item in values:
        normalized = ' '.join(str(item).split()).strip()
        key = normalized.casefold()
        if not normalized or key in seen:
            continue
        seen.add(key)
        cleaned.append(normalized)
    return cleaned


class MetadataExtraction(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    summary: str = Field(description='A concise 2-4 sentence summary.')
    regions: list[str] = Field(default_factory=list)
    varieties: list[str] = Field(default_factory=list)
    diseases: list[str] = Field(default_factory=list)
    winemaking_steps: list[str] = Field(default_factory=list)
    quality_effects: list[str] = Field(default_factory=list)
    keywords: list[str] = Field(default_factory=list)

    @field_validator(
        'regions',
        'varieties',
        'diseases',
        'winemaking_steps',
        'quality_effects',
        'keywords',
        mode='before',
    )
    @classmethod
    def ensure_list(cls, value: Any) -> list[str]:
        return clean_list(value)


class QuestionTriage(BaseModel):
    model_config = ConfigDict(str_strip_whitespace=True)

    regions: list[str] = Field(default_factory=list)
    varieties: list[str] = Field(default_factory=list)
    diseases: list[str] = Field(default_factory=list)
    winemaking_steps: list[str] = Field(default_factory=list)
    quality_effects: list[str] = Field(default_factory=list)
    keywords: list[str] = Field(default_factory=list)
    requested_outcome: Literal['diagnosis', 'quality_interpretation', 'process_advice', 'mixed']

    @field_validator(
        'regions',
        'varieties',
        'diseases',
        'winemaking_steps',
        'quality_effects',
        'keywords',
        mode='before',
    )
    @classmethod
    def ensure_list(cls, value: Any) -> list[str]:
        return clean_list(value)


class SqliteQueryPlan(BaseModel):
    sql: str
    params: list[str] = Field(default_factory=list)
    reason: str


class QdrantQueryPlan(BaseModel):
    query_text: str
    regions: list[str] = Field(default_factory=list)
    varieties: list[str] = Field(default_factory=list)
    diseases: list[str] = Field(default_factory=list)
    winemaking_steps: list[str] = Field(default_factory=list)
    quality_effects: list[str] = Field(default_factory=list)
    keywords: list[str] = Field(default_factory=list)
    reason: str

    @field_validator(
        'regions',
        'varieties',
        'diseases',
        'winemaking_steps',
        'quality_effects',
        'keywords',
        mode='before',
    )
    @classmethod
    def ensure_list(cls, value: Any) -> list[str]:
        return clean_list(value)


class FinalAnswer(BaseModel):
    answer: str



## Load Prompts
The notebook only uses `prompts/wine_demo_prompts.py`.



In [9]:
PROMPTS = runpy.run_path(str(PROJECT_ROOT / 'prompts' / 'wine_demo_prompts.py'))

DOCUMENT_METADATA_EXTRACTION_PROMPT = PROMPTS['DOCUMENT_METADATA_EXTRACTION_PROMPT']
QUESTION_TRIAGE_PROMPT = PROMPTS['QUESTION_TRIAGE_PROMPT']
FINAL_SYNTHESIS_PROMPT = PROMPTS['FINAL_SYNTHESIS_PROMPT']
render_sqlite_sql_prompt = PROMPTS['render_sqlite_sql_prompt']
render_qdrant_query_prompt = PROMPTS['render_qdrant_query_prompt']
normalize_allowed_values = PROMPTS['normalize_allowed_values']

In [10]:
def normalize_azure_openai_base_url(endpoint: str) -> str:
    normalized = endpoint.rstrip('/')
    if normalized.endswith('/openai/v1'):
        return normalized + '/'
    if normalized.endswith('/openai'):
        return normalized + '/v1/'
    if '.openai.azure.com' in normalized:
        return normalized + '/openai/v1/'
    return normalized + '/'

api_key = os.getenv('AZURE_OPENAI_API_KEY')
azure_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')
deployment = AZURE_OPENAI_DEPLOYMENT

client = AsyncOpenAI(
    base_url=normalize_azure_openai_base_url(azure_endpoint),
    api_key=api_key,
    )

chat_model = OpenAIChatModel(model_name=deployment, provider=OpenAIProvider(openai_client=client))

In [11]:
cohere_api_key = os.getenv('COHERE_API_KEY')
cohere_client = cohere.ClientV2(api_key=cohere_api_key)

## Shared Agents
Build the reusable extraction, triage, and synthesis agents once so later cells can call them directly.



In [13]:
metadata_agent = Agent(
    chat_model,
    instructions=DOCUMENT_METADATA_EXTRACTION_PROMPT,
    output_type=MetadataExtraction,
)
triage_agent = Agent(
    chat_model,
    instructions=QUESTION_TRIAGE_PROMPT,
    output_type=QuestionTriage,
)
final_answer_agent = Agent(
    chat_model,
    instructions=FINAL_SYNTHESIS_PROMPT,
    output_type=FinalAnswer,
)

## Run One Live Metadata Extraction
This is the booth-safe LLM path: one document, one extraction, visible output.



In [14]:
one_extraction = await metadata_agent.run(
    f"""
    Extract metadata from this markdown document body.

    <document_body>
    {sample_post.content}
    </document_body>
    """.strip()
)
one_extraction.output

MetadataExtraction(summary='This brief frames Botrytis cinerea in a French regional context, using Burgundy as the primary anchor because gray rot is described as a major problem on ripe grapes there, as well as in Beaujolais and the Loire Valley. It emphasizes late-season bunch rot under humid, temperate to hot conditions, with symptoms including berry softening, browning, and characteristic gray sporulation on clusters. The document also notes that losses can continue after harvest during storage or transport and cites French literature from Bordeaux linking Botrytis bunch rot to reduced yield and wine quality.', regions=['Burgundy', 'Beaujolais', 'Loire Valley', 'Bordeaux', 'France'], varieties=['Pinot Noir'], diseases=['Botrytis cinerea', 'gray mold', 'gray rot', 'Botrytis bunch rot', 'bunch rot'], winemaking_steps=['harvest', 'storage', 'transport'], quality_effects=['reduced yield', 'reduced wine quality', 'berry softening', 'browning', 'cluster deterioration', 'post-harvest loss

## Load Documents And Aggregate Metadata
The graph is built from aggregated datastore metadata, not from document nodes.



In [15]:
def build_keyword_blob(document: dict[str, Any]) -> str:
    metadata = document['metadata']
    pieces = [
        document['title'],
        document['country'],
        document['region'],
        metadata['summary'],
    ]
    for field_name, _ in TERM_FIELDS:
        pieces.extend(metadata[field_name])
    return ' | '.join(piece for piece in pieces if piece)


def load_demo_documents(data_root: Path) -> list[dict[str, Any]]:
    documents: list[dict[str, Any]] = []
    for path in sorted(data_root.rglob('*.md')):
        post = frontmatter.load(path)
        metadata = MetadataExtraction.model_validate(post.metadata['metadata']).model_dump(mode='json')
        metadata['regions'] = clean_list(
            [
                post.metadata.get('country', ''),
                post.metadata.get('region', ''),
                *metadata['regions'],
            ]
        )
        document = {
            'document_id': str(path.relative_to(PROJECT_ROOT)),
            'title': post.metadata['title'],
            'repository': post.metadata['repository'],
            'country': post.metadata.get('country', ''),
            'region': post.metadata.get('region', ''),
            'source': post.metadata.get('source', ''),
            'source_urls': list(post.metadata.get('source_urls', [])),
            'source_path': str(path.relative_to(PROJECT_ROOT)),
            'content': post.content.strip(),
            'metadata': metadata,
        }
        document['keyword_blob'] = build_keyword_blob(document)
        documents.append(document)
    return documents


def build_repository_allowed_values(documents: list[dict[str, Any]]) -> dict[str, dict[str, list[str]]]:
    values: dict[str, dict[str, set[str]]] = defaultdict(lambda: defaultdict(set))
    for document in documents:
        repository = document['repository']
        if document['country']:
            values[repository]['countries'].add(document['country'])
        if document['region']:
            values[repository]['region_labels'].add(document['region'])
        values[repository]['sources'].add(document['source'])
        for field_name, _ in TERM_FIELDS:
            values[repository][field_name].update(document['metadata'][field_name])
    return {
        repository: normalize_allowed_values(repository_values)
        for repository, repository_values in values.items()
    }


def build_coverage_rows(documents: list[dict[str, Any]]) -> list[dict[str, Any]]:
    counter: Counter[tuple[str, str, str, str]] = Counter()
    for document in documents:
        repository = document['repository']
        for field_name, label in TERM_FIELDS:
            for value in document['metadata'][field_name]:
                counter[(repository, label, value, normalize_text(value))] += 1

    rows = [
        {
            'datastore': repository,
            'term_type': label,
            'term': term,
            'term_normalized': term_normalized,
            'count': count,
        }
        for (repository, label, term, term_normalized), count in counter.items()
    ]
    return sorted(rows, key=lambda row: (row['datastore'], row['term_type'], row['term']))


def build_overlap_rows(coverage_rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    by_repository: dict[str, dict[str, dict[str, str]]] = defaultdict(lambda: defaultdict(dict))
    for row in coverage_rows:
        by_repository[row['datastore']][row['term_type']][row['term_normalized']] = row['term']

    overlaps: list[dict[str, Any]] = []
    for index, left in enumerate(REPOSITORY_ORDER):
        for right in REPOSITORY_ORDER[index + 1:]:
            shared_terms: list[str] = []
            shared_types: list[str] = []
            left_terms = by_repository[left]
            right_terms = by_repository[right]
            for term_type in sorted(set(left_terms) | set(right_terms)):
                shared_keys = sorted(set(left_terms[term_type]) & set(right_terms[term_type]))
                if not shared_keys:
                    continue
                shared_types.append(term_type)
                shared_terms.extend(left_terms[term_type][key] for key in shared_keys)
            if shared_terms:
                overlaps.append(
                    {
                        'left_datastore': left,
                        'right_datastore': right,
                        'shared_terms': shared_terms,
                        'shared_types': shared_types,
                        'score': len(shared_terms),
                    }
                )
    return overlaps


def triage_terms(triage: QuestionTriage) -> list[str]:
    return clean_list(
        triage.regions
        + triage.varieties
        + triage.diseases
        + triage.winemaking_steps
        + triage.quality_effects
        + triage.keywords
    )



In [17]:
documents = load_demo_documents(DATA_ROOT)
REPOSITORY_ALLOWED_VALUES = build_repository_allowed_values(documents)
coverage_rows = build_coverage_rows(documents)
overlap_rows = build_overlap_rows(coverage_rows)

print(f'Loaded {len(documents)} documents')
pd.DataFrame(coverage_rows).head(12)

Loaded 16 documents


,datastore,term_type,term,term_normalized,count
0,vine_diseases,Disease,Botrytis,botrytis,2
1,vine_diseases,Disease,Botrytis bunch rot,botrytis bunch rot,1
2,vine_diseases,Disease,Botrytis cinerea,botrytis cinerea,2
3,vine_diseases,Disease,black rot,black rot,2
4,vine_diseases,Disease,downy mildew,downy mildew,4
5,vine_diseases,Disease,excoriose,excoriose,1
6,vine_diseases,Disease,frost damage,frost damage,2
7,vine_diseases,Disease,fungal diseases,fungal diseases,1
8,vine_diseases,Disease,gray mold,gray mold,1
9,vine_diseases,Disease,gray rot,gray rot,2


In [18]:
pd.DataFrame(overlap_rows)

,left_datastore,right_datastore,shared_terms,shared_types,score
0,vine_diseases,wine_diseases,"[Botrytis bunch rot, Botrytis cinerea, gray ro...","[Disease, Keyword, QualityEffect, Region, Vari...",19
1,vine_diseases,wine_making,"[Botrytis cinerea, gray rot, noble rot, Centra...","[Disease, Region, Variety]",10
2,wine_diseases,wine_making,"[Botrytis cinerea, gray rot, noble rot, botryt...","[Disease, Keyword, Region, Variety]",14


## Batch Extraction Checkpoint
The frontmatter already contains metadata.
This optional rebuild path only reruns extraction if you want to refresh the JSONL checkpoints offline.



In [ ]:
extraction_records: list[dict[str, Any]] = []

with JSONL_PATH.open('w', encoding='utf-8') as jsonl_file:
    for document in documents:
        result = await metadata_agent.run(
            f"""
            Extract metadata from this markdown document body.

            <document_body>
            {document['content']}
            </document_body>
            """.strip()
        )
        record = {
            'document_id': document['document_id'],
            'repository': document['repository'],
            'source_path': document['source_path'],
            'metadata': result.output.model_dump(mode='json'),
        }
        jsonl_file.write(json.dumps(record, ensure_ascii=False) + '\n')
        jsonl_file.flush()
        extraction_records.append(record)

print(f'Wrote {len(extraction_records)} checkpoint records to {JSONL_PATH}')

## SQLite Datastore 1: `vine_diseases`
Seed the vineyard disease store directly from the curated markdown and stored metadata.
The list-like metadata fields are kept as readable `*_csv` payload columns because SQLite never queries inside them.



In [24]:
VINE_DISEASES_DB_PATH = SQLITE_DIR / 'vine_diseases.db'

In [ ]:
vine_diseases_documents = [document for document in documents if document['repository'] == 'vine_diseases']

VINE_DISEASES_SCHEMA_SQL = """
CREATE TABLE documents (
  rowid INTEGER PRIMARY KEY AUTOINCREMENT,
  document_id TEXT UNIQUE NOT NULL,
  title TEXT NOT NULL,
  repository TEXT NOT NULL,
  country TEXT,
  region TEXT,
  source TEXT,
  source_path TEXT,
  summary TEXT,
  body TEXT,
  regions_csv TEXT,
  varieties_csv TEXT,
  diseases_csv TEXT,
  winemaking_steps_csv TEXT,
  quality_effects_csv TEXT,
  keywords_csv TEXT,
  keyword_blob TEXT
);

CREATE VIRTUAL TABLE documents_fts USING fts5(
  title,
  body,
  summary,
  keyword_blob,
  content='documents',
  content_rowid='rowid'
);
""".strip()

vine_diseases_seed_db = sqlite3.connect(VINE_DISEASES_DB_PATH)
vine_diseases_seed_db.executescript(
    """
    PRAGMA foreign_keys = ON;
    DROP TABLE IF EXISTS documents_fts;
    DROP TABLE IF EXISTS documents;
    """
)
vine_diseases_seed_db.executescript(VINE_DISEASES_SCHEMA_SQL)

for document in vine_diseases_documents:
    metadata = document['metadata']
    cursor = vine_diseases_seed_db.execute(
        """
        INSERT INTO documents (
            document_id,
            title,
            repository,
            country,
            region,
            source,
            source_path,
            summary,
            body,
            regions_csv,
            varieties_csv,
            diseases_csv,
            winemaking_steps_csv,
            quality_effects_csv,
            keywords_csv,
            keyword_blob
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            document['document_id'],
            document['title'],
            document['repository'],
            document['country'],
            document['region'],
            document['source'],
            document['source_path'],
            metadata['summary'],
            document['content'],
            ', '.join(metadata['regions']),
            ', '.join(metadata['varieties']),
            ', '.join(metadata['diseases']),
            ', '.join(metadata['winemaking_steps']),
            ', '.join(metadata['quality_effects']),
            ', '.join(metadata['keywords']),
            document['keyword_blob'],
        ),
    )
    rowid = cursor.lastrowid
    vine_diseases_seed_db.execute(
        'INSERT INTO documents_fts(rowid, title, body, summary, keyword_blob) VALUES (?, ?, ?, ?, ?)',
        (
            rowid,
            document['title'],
            document['content'],
            metadata['summary'],
            document['keyword_blob'],
        ),
    )

vine_diseases_seed_db.commit()
vine_diseases_seed_db.close()

{
    'db_path': str(VINE_DISEASES_DB_PATH),
    'documents_seeded': len(vine_diseases_documents),
}



## SQLite Datastore 2: `wine_diseases`
Seed the wine quality store separately so the second datastore is visible as its own organism.
The list-like metadata fields stay as readable `*_csv` payload columns here as well.



In [23]:
WINE_DISEASES_DB_PATH = SQLITE_DIR / 'wine_diseases.db'

In [ ]:
wine_diseases_documents = [document for document in documents if document['repository'] == 'wine_diseases']

WINE_DISEASES_SCHEMA_SQL = """
CREATE TABLE documents (
  rowid INTEGER PRIMARY KEY AUTOINCREMENT,
  document_id TEXT UNIQUE NOT NULL,
  title TEXT NOT NULL,
  repository TEXT NOT NULL,
  country TEXT,
  region TEXT,
  source TEXT,
  source_path TEXT,
  summary TEXT,
  body TEXT,
  regions_csv TEXT,
  varieties_csv TEXT,
  diseases_csv TEXT,
  winemaking_steps_csv TEXT,
  quality_effects_csv TEXT,
  keywords_csv TEXT,
  keyword_blob TEXT
);

CREATE VIRTUAL TABLE documents_fts USING fts5(
  title,
  body,
  summary,
  keyword_blob,
  content='documents',
  content_rowid='rowid'
);
""".strip()

wine_diseases_seed_db = sqlite3.connect(WINE_DISEASES_DB_PATH)
wine_diseases_seed_db.executescript(
    """
    PRAGMA foreign_keys = ON;
    DROP TABLE IF EXISTS documents_fts;
    DROP TABLE IF EXISTS documents;
    """
)
wine_diseases_seed_db.executescript(WINE_DISEASES_SCHEMA_SQL)

for document in wine_diseases_documents:
    metadata = document['metadata']
    cursor = wine_diseases_seed_db.execute(
        """
        INSERT INTO documents (
            document_id,
            title,
            repository,
            country,
            region,
            source,
            source_path,
            summary,
            body,
            regions_csv,
            varieties_csv,
            diseases_csv,
            winemaking_steps_csv,
            quality_effects_csv,
            keywords_csv,
            keyword_blob
        ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            document['document_id'],
            document['title'],
            document['repository'],
            document['country'],
            document['region'],
            document['source'],
            document['source_path'],
            metadata['summary'],
            document['content'],
            ', '.join(metadata['regions']),
            ', '.join(metadata['varieties']),
            ', '.join(metadata['diseases']),
            ', '.join(metadata['winemaking_steps']),
            ', '.join(metadata['quality_effects']),
            ', '.join(metadata['keywords']),
            document['keyword_blob'],
        ),
    )
    rowid = cursor.lastrowid
    wine_diseases_seed_db.execute(
        'INSERT INTO documents_fts(rowid, title, body, summary, keyword_blob) VALUES (?, ?, ?, ?, ?)',
        (
            rowid,
            document['title'],
            document['content'],
            metadata['summary'],
            document['keyword_blob'],
        ),
    )

wine_diseases_seed_db.commit()
wine_diseases_seed_db.close()

{
    'db_path': str(WINE_DISEASES_DB_PATH),
    'documents_seeded': len(wine_diseases_documents),
}



## Load Existing SQLite Datastores
Open each SQLite file separately so the rest of the notebook can start from the existing stores.



In [25]:
vine_diseases_db = sqlite3.connect(VINE_DISEASES_DB_PATH)
vine_diseases_db.row_factory = sqlite3.Row
vine_diseases_schema_rows = vine_diseases_db.execute(
    """
    SELECT sql
    FROM sqlite_master
    WHERE type IN ('table', 'view')
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """
).fetchall()
vine_diseases_schema_sql = '\n\n'.join(row['sql'] for row in vine_diseases_schema_rows if row['sql'])

{
    'db_path': str(VINE_DISEASES_DB_PATH),
    'tables': [row['name'] for row in vine_diseases_db.execute("SELECT name FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY name").fetchall()],
    'documents': vine_diseases_db.execute('SELECT COUNT(*) AS count FROM documents').fetchone()['count'],
}



{'db_path': '/Users/konrad/Projects/building-knowledge-graphs/processed/winde_demo/sqlite/vine_diseases.db',
 'tables': ['documents',
  'documents_fts',
  'documents_fts_config',
  'documents_fts_data',
  'documents_fts_docsize',
  'documents_fts_idx',
  'sqlite_sequence'],
 'documents': 6}

In [26]:
wine_diseases_db = sqlite3.connect(WINE_DISEASES_DB_PATH)
wine_diseases_db.row_factory = sqlite3.Row
wine_diseases_schema_rows = wine_diseases_db.execute(
    """
    SELECT sql
    FROM sqlite_master
    WHERE type IN ('table', 'view')
      AND name NOT LIKE 'sqlite_%'
    ORDER BY name
    """
).fetchall()
wine_diseases_schema_sql = '\n\n'.join(row['sql'] for row in wine_diseases_schema_rows if row['sql'])

{
    'db_path': str(WINE_DISEASES_DB_PATH),
    'tables': [row['name'] for row in wine_diseases_db.execute("SELECT name FROM sqlite_master WHERE type IN ('table', 'view') ORDER BY name").fetchall()],
    'documents': wine_diseases_db.execute('SELECT COUNT(*) AS count FROM documents').fetchone()['count'],
}



{'db_path': '/Users/konrad/Projects/building-knowledge-graphs/processed/winde_demo/sqlite/wine_diseases.db',
 'tables': ['documents',
  'documents_fts',
  'documents_fts_config',
  'documents_fts_data',
  'documents_fts_docsize',
  'documents_fts_idx',
  'sqlite_sequence'],
 'documents': 5}

In [27]:
vine_diseases_demo_row = dict(
    vine_diseases_db.execute(
        """
        SELECT
          d.document_id,
          d.title,
          d.region,
          d.summary,
          d.source_path,
          snippet(documents_fts, 1, '[', ']', '...', 18) AS snippet,
          bm25(documents_fts) AS score
        FROM documents_fts
        JOIN documents d ON d.rowid = documents_fts.rowid
        WHERE documents_fts MATCH ?
          AND d.country = ?
        ORDER BY score
        LIMIT 1
        """,
        ('central OR Poland OR cultivars OR fungal', 'Poland'),
    ).fetchone()
)
vine_diseases_demo_row



{'document_id': 'data/winde_demo/vine_diseases/poland_processing_cultivars.md',
 'title': 'Central Poland brief: why processing cultivars need both fungal tolerance and winter hardiness',
 'region': 'Central Poland',
 'summary': 'Central Poland is a strong vineyard-routing example because disease pressure cannot be separated from cold-climate limits. The Polish cultivar studies show that interspecific hybrids often outperform classic Vitis vinifera cultivars when winter hardiness, fungal tolerance, and processing suitability are all considered together.',
 'source_path': 'data/winde_demo/vine_diseases/poland_processing_cultivars.md',
 'snippet': '...[cultivars]. That matters enormously for routing. If a user asks, “What looks safer in [central] [Poland] when [fungal]...',
 'score': -1.2438385844294306}

In [28]:
wine_diseases_demo_row = dict(
    wine_diseases_db.execute(
        """
        SELECT
          d.document_id,
          d.title,
          d.region,
          d.summary,
          d.source_path,
          snippet(documents_fts, 1, '[', ']', '...', 18) AS snippet,
          bm25(documents_fts) AS score
        FROM documents_fts
        JOIN documents d ON d.rowid = documents_fts.rowid
        WHERE documents_fts MATCH ?
          AND d.country = ?
        ORDER BY score
        LIMIT 1
        """,
        ('gray OR rot OR laccase OR Bordeaux', 'France'),
    ).fetchone()
)
wine_diseases_demo_row



{'document_id': 'data/winde_demo/wine_diseases/france_gray_rot_quality_damage.md',
 'title': 'Bordeaux brief: why gray rot becomes a wine-quality problem once infected fruit reaches the cellar',
 'region': 'Bordeaux',
 'summary': 'INRAE work from French oenology groups treats Botrytis cinerea as a direct wine-quality problem once infected berries enter must preparation, especially in red wines where laccase-driven oxidation can degrade colour and destabilize phenolic balance. Bordeaux is the anchor region for this brief because the research focus on botrytised grapes, oxidation damage, sulphur dioxide, and tannin-based mitigation gives the demo a clear post-harvest route into the wine_diseases datastore.',
 'source_path': 'data/winde_demo/wine_diseases/france_gray_rot_quality_damage.md',
 'snippet': '...The graph only needs to know that this datastore covers France, [Bordeaux], [gray] [rot], [laccase], and colour degradation...',
 'score': -0.6729401952021643}

## Create semantic layer

In [29]:
def validate_sqlite_select(sql: str) -> None:
    normalized = sql.strip().lower()
    if not normalized.startswith('select'):
        raise ValueError(f'SQLite tool must emit a SELECT query, got: {sql}')
    forbidden_tokens = [' insert ', ' update ', ' delete ', ' drop ', ' alter ', ' pragma ', ' attach ', ' detach ']
    padded = f' {normalized} '
    for token in forbidden_tokens:
        if token in padded:
            raise ValueError(f'Forbidden SQL token in generated query: {token.strip()}')



In [30]:
async def sqlite_tool(question: str, triage: QuestionTriage, repository: str) -> dict[str, Any]:
    if repository == 'vine_diseases':
        db_path = VINE_DISEASES_DB_PATH
        schema_sql = vine_diseases_schema_sql
    elif repository == 'wine_diseases':
        db_path = WINE_DISEASES_DB_PATH
        schema_sql = wine_diseases_schema_sql
    else:
        raise ValueError(f'Unknown SQLite datastore: {repository}')

    allowed_values = REPOSITORY_ALLOWED_VALUES[repository]
    prompt = render_sqlite_sql_prompt(schema_sql, allowed_values)
    agent = Agent(
        chat_model,
        instructions=prompt,
        output_type=SqliteQueryPlan,
    )
    request = f"""
    User question:
    {question}

    Normalized triage:
    {triage.model_dump_json(indent=2)}

    Datastore:
    {repository}
    """.strip()
    plan = (await agent.run(request)).output
    validate_sqlite_select(plan.sql)

    connection = sqlite3.connect(db_path)
    connection.row_factory = sqlite3.Row
    rows = [dict(row) for row in connection.execute(plan.sql, plan.params).fetchall()]
    if not rows:
        fallback_terms = clean_list(
            triage.diseases
            + triage.varieties
            + triage.winemaking_steps
            + triage.quality_effects
            + triage.keywords
        )
        fallback_match = ' OR '.join(
            f'"{term}"' if ' ' in term else term
            for term in fallback_terms[:6]
        )
        fallback_sql = """
        SELECT
          d.document_id,
          d.title,
          d.summary,
          d.source_path,
          d.country,
          d.region,
          snippet(documents_fts, 1, '[', ']', '...', 18) AS snippet,
          bm25(documents_fts) AS score
        FROM documents_fts
        JOIN documents d ON d.rowid = documents_fts.rowid
        WHERE documents_fts MATCH ?
        """.strip()
        fallback_params: list[str] = [fallback_match or question]
        country_values = REPOSITORY_ALLOWED_VALUES[repository].get('countries', [])
        region_values = REPOSITORY_ALLOWED_VALUES[repository].get('region_labels', [])
        for region in triage.regions:
            if region in country_values:
                fallback_sql += '\n  AND d.country = ?'
                fallback_params.append(region)
                break
            if region in region_values:
                fallback_sql += '\n  AND d.region = ?'
                fallback_params.append(region)
                break
        fallback_sql += '\nORDER BY score\nLIMIT 1'
        rows = [dict(row) for row in connection.execute(fallback_sql, fallback_params).fetchall()]
    connection.close()
    top_result = rows[0] if rows else None
    if top_result:
        parts = [
            f"Title: {top_result.get('title', '')}",
            f"Region: {top_result.get('region', '')}",
            f"Source path: {top_result.get('source_path', '')}",
            f"Summary: {top_result.get('summary', '')}",
        ]
        if top_result.get('snippet'):
            parts.append(f"Snippet: {top_result['snippet']}")
        text = '\n'.join(part for part in parts if part and not part.endswith(': '))
    else:
        text = 'No result.'
    return {
        'datastore': repository,
        'query_plan': plan.model_dump(mode='json'),
        'top_result': top_result,
        'text': text,
    }



In [31]:
manual_vine_question = 'Which cultivars look safest for central Poland if fungal diseases are the main constraint?'
manual_vine_triage = QuestionTriage(
    regions=['Central Poland', 'Poland'],
    varieties=[],
    diseases=['fungal diseases'],
    winemaking_steps=[],
    quality_effects=[],
    keywords=['cultivars', 'fungal tolerance', 'winter hardiness'],
    requested_outcome='diagnosis',
)
manual_vine_result = await sqlite_tool(manual_vine_question, manual_vine_triage, 'vine_diseases')

{
    'question': manual_vine_question,
    'triage': manual_vine_triage.model_dump(mode='json'),
    'sql': manual_vine_result['query_plan']['sql'],
    'params': manual_vine_result['query_plan']['params'],
    'top_result_title': manual_vine_result['top_result']['title'] if manual_vine_result['top_result'] else None,
    'text': manual_vine_result['text'],
}


{'question': 'Which cultivars look safest for central Poland if fungal diseases are the main constraint?',
 'triage': {'regions': ['Central Poland', 'Poland'],
  'varieties': [],
  'diseases': ['fungal diseases'],
  'winemaking_steps': [],
  'quality_effects': [],
  'keywords': ['cultivars', 'fungal tolerance', 'winter hardiness'],
  'requested_outcome': 'diagnosis'},
 'sql': "SELECT d.document_id, d.title, d.summary, d.source_path, d.country, d.region, snippet(documents_fts, 1, '[', ']', '...', 18) AS snippet, bm25(documents_fts) AS score\nFROM documents_fts\nJOIN documents d ON d.rowid = documents_fts.rowid\nWHERE documents_fts MATCH ?\n  AND d.region = ?\nORDER BY bm25(documents_fts)\nLIMIT 1;",
 'params': ['"fungal diseases" OR "fungal tolerance" OR "winter hardiness"',
  'Central Poland'],
 'top_result_title': 'Central Poland brief: why processing cultivars need both fungal tolerance and winter hardiness',
 'text': 'Title: Central Poland brief: why processing cultivars need both f

## Qdrant Path For `wine_making`
Use a local Qdrant running in Docker, chunk the cellar-process documents, extract chunk metadata, then load the existing collection again for retrieval.
The chunking, extraction, and upsert cells are precompute cells; the live booth path can jump straight to loading the existing collection.



### Chunk `wine_making` Documents
Split the text with LangChain's `RecursiveCharacterTextSplitter` and keep parent-document plus chunk-number metadata.



In [ ]:
wine_making_documents = [document for document in documents if document['repository'] == 'wine_making']
wine_making_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
wine_making_chunks: list[dict[str, Any]] = []

for document in wine_making_documents:
    chunk_texts = wine_making_splitter.split_text(document['content'])
    for chunk_number, chunk_text in enumerate(chunk_texts, start=1):
        wine_making_chunks.append(
            {
                'chunk_id': f"{document['document_id']}::chunk::{chunk_number}",
                'document_id': document['document_id'],
                'chunk_number': chunk_number,
                'repository': document['repository'],
                'title': document['title'],
                'text': chunk_text,
                'metadata': {
                    **document['metadata'],
                    'country': document['country'],
                    'region_label': document['region'],
                    'source': document['source'],
                    'source_path': document['source_path'],
                    'parent_document_id': document['document_id'],
                    'parent_title': document['title'],
                    'chunk_number': chunk_number,
                },
            }
        )

pd.DataFrame(
    {
        'chunk_id': chunk['chunk_id'],
        'parent_document_id': chunk['document_id'],
        'chunk_number': chunk['chunk_number'],
        'title': chunk['title'],
        'length': len(chunk['text']),
    }
    for chunk in wine_making_chunks[:10]
)



### Extract Metadata For The Chunks
Each chunk gets its own metadata extraction while still keeping the parent-document and chunk-number trace.



In [ ]:
enriched_wine_making_chunks: list[dict[str, Any]] = []

with CHUNK_JSONL_PATH.open('w', encoding='utf-8') as jsonl_file:
    for chunk in wine_making_chunks:
        result = await metadata_agent.run(
            f"""
            Extract metadata from this markdown document body.

            <document_body>
            {chunk['text']}
            </document_body>
            """.strip()
        )
        enriched = {
            **chunk,
            'metadata': {
                **chunk['metadata'],
                **result.output.model_dump(mode='json'),
            },
        }
        jsonl_file.write(json.dumps(enriched, ensure_ascii=False) + '\n')
        jsonl_file.flush()
        enriched_wine_making_chunks.append(enriched)

print(f'Wrote {len(enriched_wine_making_chunks)} chunk records to {CHUNK_JSONL_PATH}')



[JSONL wine making chunks](processed/winde_demo/wine_making_chunks.jsonl)

### Seed The Local Qdrant Collection
This is the indexing pass against the local Docker-backed Qdrant.



In [32]:
cohere_client.embed(input_type="search_document",model=COHERE_EMBED_MODEL,texts=[sample_post.content])

EmbedByTypeResponse(response_type='embeddings_by_type', id='fb580b51-e40e-4de9-a6ae-6121128fb5b5', embeddings=EmbedByTypeResponseEmbeddings(float_=[[0.02854904, 0.003931809, 0.020085387, -0.00071056787, -0.021222295, -0.040170774, -0.0007184631, -0.01629569, 0.03966548, -0.018822154, -0.03208609, -0.017306276, -0.048508104, -0.04749752, -0.00062766834, 0.014527166, 0.0062529976, 0.007674133, -0.0033949357, -0.0067898715, 0.019201124, -0.040928714, -0.013074449, 0.0021632845, 0.026654191, -0.060129832, 0.0030475468, -0.005810866, 0.02463302, -0.008842623, -0.011937541, 0.044718407, -0.046992227, -0.0294333, -0.046234284, -0.014400843, -0.0041055037, -0.018443184, -0.0037896954, -0.0080531025, 0.02703316, -0.0057161236, -0.022359204, -0.054571617, 0.041939296, -0.018822154, 0.076299205, 0.0052108313, -0.036886368, -0.030822858, 0.039160185, 0.052550443, -0.008716299, -0.007358325, -0.010674309, 0.042697236, 0.02981227, -0.034865197, -0.01667466, 0.039412834, -0.028675362, 0.005400316, -0

In [ ]:
embedding_response = cohere_client.embed(
    model=COHERE_EMBED_MODEL,
    texts=[chunk["text"] for chunk in enriched_wine_making_chunks],
    input_type="search_document",
    output_dimension=seed_qdrant_vector_size,
    embedding_types=["float"],
)

wine_making_vectors = extract_float_embeddings(embedding_response)
if len(wine_making_vectors[0]) != seed_qdrant_vector_size:
    raise RuntimeError(
        f'Qdrant seed vector size mismatch for `{QDRANT_COLLECTION}`: collection expects {seed_qdrant_vector_size}, '
        f'but `{COHERE_EMBED_MODEL}` returned {len(wine_making_vectors[0])}. '
        'Recreate the collection or set WINE_DEMO_QDRANT_VECTOR_SIZE to the existing collection size.'
    )

In [ ]:
qdrant_point_ids = list(range(1, len(enriched_wine_making_chunks) + 1))

In [ ]:
if QDRANT_COLLECTION not in existing_qdrant_collections:
    qdrant_seed_client.create_collection(
        collection_name=QDRANT_COLLECTION,
        vectors_config=models.VectorParams(
            size=seed_qdrant_vector_size,
            distance=models.Distance.COSINE,
        ),
    )

In [ ]:
qdrant_seed_client.upsert(
    collection_name=QDRANT_COLLECTION,
    wait=True,
    points=models.Batch(
        ids=qdrant_point_ids,
        vectors=wine_making_vectors,
        payloads=[
            {
                "chunk_id": chunk["chunk_id"],
                "text": chunk["text"],
                "title": chunk["title"],
                "document_id": chunk["document_id"],
                "chunk_number": chunk["chunk_number"],
                "metadata": chunk["metadata"],
            }
            for chunk in enriched_wine_making_chunks
        ],
    ),
)


### Load The Existing Qdrant Collection
Open a fresh client against the same collection so retrieval can start from the already-indexed state.
If the collection was already built earlier, this is the entry point for the live demo.



In [58]:
qdrant_query_client = QdrantClient(url=QDRANT_URL)

In [59]:
qdrant_query_client.get_collection(collection_name=QDRANT_COLLECTION)


CollectionInfo(status=<CollectionStatus.GREEN: 'green'>, optimizer_status=<OptimizersStatusOneOf.OK: 'ok'>, warnings=None, indexed_vectors_count=0, points_count=58, segments_count=7, config=CollectionConfig(params=CollectionParams(vectors=VectorParams(size=1536, distance=<Distance.COSINE: 'Cosine'>, hnsw_config=None, quantization_config=None, on_disk=None, datatype=None, multivector_config=None), shard_number=1, sharding_method=None, replication_factor=1, write_consistency_factor=1, read_fan_out_factor=None, read_fan_out_delay_ms=None, on_disk_payload=True, sparse_vectors=None), hnsw_config=HnswConfig(m=16, ef_construct=100, full_scan_threshold=10000, max_indexing_threads=0, on_disk=False, payload_m=None, inline_storage=None), optimizer_config=OptimizersConfig(deleted_threshold=0.2, vacuum_min_vector_number=1000, default_segment_number=0, max_segment_size=None, memmap_threshold=None, indexing_threshold=10000, flush_interval_sec=5, max_optimization_threads=None, prevent_unoptimized=None

In [50]:
async def qdrant_tool(question: str, triage: QuestionTriage) -> dict[str, Any]:
    if 'qdrant_query_client' not in globals():
        raise RuntimeError('Run the existing Qdrant collection load cell before using the wine_making retrieval tool.')

    prompt = render_qdrant_query_prompt(REPOSITORY_ALLOWED_VALUES['wine_making'])
    agent = Agent(
        chat_model,
        instructions=prompt,
        output_type=QdrantQueryPlan,
    )

    request = f"""
    User question:
    {question}

    Normalized triage:
    {triage.model_dump_json(indent=2)}
    """.strip()
    plan = (await agent.run(request)).output

    collection_info = qdrant_query_client.get_collection(QDRANT_COLLECTION)
    vector_config = collection_info.config.params.vectors
    if hasattr(vector_config, 'size'):
        qdrant_vector_size = vector_config.size
    else:
        qdrant_vector_size = next(iter(vector_config.values())).size

    query_vector = cohere_client.embed(
        model=COHERE_EMBED_MODEL,
        input_type="search_query",
        texts=[plan.query_text],
        output_dimension=qdrant_vector_size,
        embedding_types=["float"],
    ).embeddings.float_[0]

    must_conditions: list[models.Condition] = []
    for field_name in ['regions', 'varieties', 'diseases', 'winemaking_steps', 'quality_effects', 'keywords']:
        values = getattr(plan, field_name)
        if not values:
            continue
        must_conditions.append(
            models.FieldCondition(
                key=f'metadata.{field_name}',
                match=models.MatchAny(any=values),
            )
        )

    query_filter = models.Filter(must=must_conditions) if must_conditions else None

    try:
        result = qdrant_query_client.query_points(
            collection_name=QDRANT_COLLECTION,
            query=query_vector,
            limit=1,
            with_payload=True,
            query_filter=query_filter,
        )
    except Exception as exc:
        raise RuntimeError(
            f'Qdrant query failed against `{QDRANT_COLLECTION}` at {QDRANT_URL}. '
            f'Expected vector size {qdrant_vector_size}. Underlying error: {exc}'
        ) from exc

    top_point = result.points[0] if result.points else None
    if top_point is not None:
        payload = top_point.payload or {}
        metadata = payload.get('metadata', {})
        text = '\n'.join(
            part
            for part in [
                f"Title: {payload.get('title', '')}",
                f"Chunk number: {payload.get('chunk_number', '')}",
                f"Source path: {metadata.get('source_path', '')}",
                f"Text: {payload.get('text', '')}",
            ]
            if part and not part.endswith(': ')
        )
    else:
        text = 'No result.'

    return {
        'datastore': 'wine_making',
        'text': text,
        'plan': plan.model_dump(mode='json'),
    }

## Neo4j Aura Graph
Neo4j stores only aggregated datastore coverage and datastore overlap.



In [36]:
DATASTORE_COVERAGE_CYPHER = """
MATCH (d:DataStore {demo_id: $demo_id})-[r:COVERS]->(c)
WHERE c.name_normalized IN $terms
RETURN d.name AS datastore, labels(c)[0] AS concept_type, c.name AS matched_term, r.count AS coverage_count
ORDER BY coverage_count DESC, datastore, concept_type, matched_term
""".strip()

DATASTORE_OVERLAP_CYPHER = """
MATCH (left:DataStore {demo_id: $demo_id})-[r:OVERLAPS_WITH]->(right:DataStore {demo_id: $demo_id})
RETURN left.name AS left_datastore, right.name AS right_datastore, r.score AS score, r.shared_types AS shared_types, r.shared_terms AS shared_terms
ORDER BY score DESC, left_datastore, right_datastore
""".strip()


def reset_demo_graph(driver) -> None:
    driver.execute_query(
        'MATCH (n {demo_id: $demo_id}) DETACH DELETE n',
        routing_=RoutingControl.WRITE,
        demo_id=DEMO_ID,
    )


def ingest_demo_graph(driver, coverage_rows: list[dict[str, Any]], overlap_rows: list[dict[str, Any]]) -> None:
    datastore_rows = [
        {
            'name': repository,
            'description': STORE_DESCRIPTIONS[repository],
        }
        for repository in REPOSITORY_ORDER
    ]
    driver.execute_query(
        """
        UNWIND $rows AS row
        MERGE (d:DataStore {demo_id: $demo_id, name: row.name})
        SET d.description = row.description
        """,
        routing_=RoutingControl.WRITE,
        demo_id=DEMO_ID,
        rows=datastore_rows,
    )

    for label in ['Region', 'Variety', 'Disease', 'Process', 'QualityEffect', 'Keyword']:
        rows = [row for row in coverage_rows if row['term_type'] == label]
        if not rows:
            continue
        query = f"""
        UNWIND $rows AS row
        MATCH (d:DataStore {{demo_id: $demo_id, name: row.datastore}})
        MERGE (c:{label} {{demo_id: $demo_id, name_normalized: row.term_normalized}})
        SET c.name = row.term
        MERGE (d)-[r:COVERS {{demo_id: $demo_id, term_type: row.term_type, term_normalized: row.term_normalized}}]->(c)
        SET r.count = row.count
        """
        driver.execute_query(
            query,
            routing_=RoutingControl.WRITE,
            demo_id=DEMO_ID,
            rows=rows,
        )

    overlap_edge_rows: list[dict[str, Any]] = []
    for row in overlap_rows:
        overlap_edge_rows.append(row)
        overlap_edge_rows.append(
            {
                'left_datastore': row['right_datastore'],
                'right_datastore': row['left_datastore'],
                'shared_terms': row['shared_terms'],
                'shared_types': row['shared_types'],
                'score': row['score'],
            }
        )

    driver.execute_query(
        """
        UNWIND $rows AS row
        MATCH (left:DataStore {demo_id: $demo_id, name: row.left_datastore})
        MATCH (right:DataStore {demo_id: $demo_id, name: row.right_datastore})
        MERGE (left)-[r:OVERLAPS_WITH {demo_id: $demo_id, target: row.right_datastore}]->(right)
        SET r.shared_terms = row.shared_terms,
            r.shared_types = row.shared_types,
            r.score = row.score
        """,
        routing_=RoutingControl.WRITE,
        demo_id=DEMO_ID,
        rows=overlap_edge_rows,
    )


def route_datastores_with_neo4j(triage: QuestionTriage) -> list[dict[str, Any]]:
    driver = neo4j_driver
    requested_terms = [normalize_text(term) for term in triage_terms(triage)]
    if not requested_terms:
        return []
    records = driver.execute_query(
        """
        MATCH (d:DataStore {demo_id: $demo_id})-[r:COVERS]->(c)
        WHERE c.name_normalized IN $terms
        WITH d, collect(DISTINCT c.name) AS matched_terms, collect(DISTINCT labels(c)[0]) AS matched_types, sum(coalesce(r.count, 1)) AS score
        RETURN d.name AS datastore, matched_terms, matched_types, score
        ORDER BY score DESC, datastore
        """,
        routing_=RoutingControl.READ,
        demo_id=DEMO_ID,
        terms=requested_terms,
    ).records
    ranked = [record.data() for record in records]
    desired_count = 2 if triage.requested_outcome == 'mixed' else 1
    return ranked[:desired_count]



### Load Aggregated Metadata For Neo4j From Files
Duplicate the frontmatter-loading logic here so the graph section can be run from scratch.



In [37]:
neo4j_documents: list[dict[str, Any]] = []

for path in sorted((DATA_ROOT / 'vine_diseases').glob('*.md')):
    post = frontmatter.load(path)
    metadata = MetadataExtraction.model_validate(post.metadata['metadata']).model_dump(mode='json')
    metadata['regions'] = clean_list([post.metadata.get('country', ''), post.metadata.get('region', ''), *metadata['regions']])
    neo4j_documents.append(
        {
            'document_id': str(path.relative_to(PROJECT_ROOT)),
            'title': post.metadata['title'],
            'repository': post.metadata['repository'],
            'country': post.metadata.get('country', ''),
            'region': post.metadata.get('region', ''),
            'source': post.metadata.get('source', ''),
            'source_urls': list(post.metadata.get('source_urls', [])),
            'source_path': str(path.relative_to(PROJECT_ROOT)),
            'content': post.content.strip(),
            'metadata': metadata,
            'keyword_blob': ' | '.join(
                piece
                for piece in [
                    post.metadata['title'],
                    post.metadata.get('country', ''),
                    post.metadata.get('region', ''),
                    metadata['summary'],
                    *metadata['regions'],
                    *metadata['varieties'],
                    *metadata['diseases'],
                    *metadata['winemaking_steps'],
                    *metadata['quality_effects'],
                    *metadata['keywords'],
                ]
                if piece
            ),
        }
    )

for path in sorted((DATA_ROOT / 'wine_diseases').glob('*.md')):
    post = frontmatter.load(path)
    metadata = MetadataExtraction.model_validate(post.metadata['metadata']).model_dump(mode='json')
    metadata['regions'] = clean_list([post.metadata.get('country', ''), post.metadata.get('region', ''), *metadata['regions']])
    neo4j_documents.append(
        {
            'document_id': str(path.relative_to(PROJECT_ROOT)),
            'title': post.metadata['title'],
            'repository': post.metadata['repository'],
            'country': post.metadata.get('country', ''),
            'region': post.metadata.get('region', ''),
            'source': post.metadata.get('source', ''),
            'source_urls': list(post.metadata.get('source_urls', [])),
            'source_path': str(path.relative_to(PROJECT_ROOT)),
            'content': post.content.strip(),
            'metadata': metadata,
            'keyword_blob': ' | '.join(
                piece
                for piece in [
                    post.metadata['title'],
                    post.metadata.get('country', ''),
                    post.metadata.get('region', ''),
                    metadata['summary'],
                    *metadata['regions'],
                    *metadata['varieties'],
                    *metadata['diseases'],
                    *metadata['winemaking_steps'],
                    *metadata['quality_effects'],
                    *metadata['keywords'],
                ]
                if piece
            ),
        }
    )

for path in sorted((DATA_ROOT / 'wine_making').glob('*.md')):
    post = frontmatter.load(path)
    metadata = MetadataExtraction.model_validate(post.metadata['metadata']).model_dump(mode='json')
    metadata['regions'] = clean_list([post.metadata.get('country', ''), post.metadata.get('region', ''), *metadata['regions']])
    neo4j_documents.append(
        {
            'document_id': str(path.relative_to(PROJECT_ROOT)),
            'title': post.metadata['title'],
            'repository': post.metadata['repository'],
            'country': post.metadata.get('country', ''),
            'region': post.metadata.get('region', ''),
            'source': post.metadata.get('source', ''),
            'source_urls': list(post.metadata.get('source_urls', [])),
            'source_path': str(path.relative_to(PROJECT_ROOT)),
            'content': post.content.strip(),
            'metadata': metadata,
            'keyword_blob': ' | '.join(
                piece
                for piece in [
                    post.metadata['title'],
                    post.metadata.get('country', ''),
                    post.metadata.get('region', ''),
                    metadata['summary'],
                    *metadata['regions'],
                    *metadata['varieties'],
                    *metadata['diseases'],
                    *metadata['winemaking_steps'],
                    *metadata['quality_effects'],
                    *metadata['keywords'],
                ]
                if piece
            ),
        }
    )

neo4j_coverage_rows = build_coverage_rows(neo4j_documents)
neo4j_overlap_rows = build_overlap_rows(neo4j_coverage_rows)
pd.DataFrame(neo4j_coverage_rows).groupby(['datastore', 'term_type']).size().rename('rows').reset_index()



,datastore,term_type,rows
0,vine_diseases,Disease,12
1,vine_diseases,Keyword,35
2,vine_diseases,Process,1
3,vine_diseases,QualityEffect,15
4,vine_diseases,Region,18
5,vine_diseases,Variety,26
6,wine_diseases,Disease,5
7,wine_diseases,Keyword,25
8,wine_diseases,Process,14
9,wine_diseases,QualityEffect,20


## Neo4j Aura Graph
Create the aggregated graph from the frontmatter-derived metadata.



In [60]:
neo4j_driver = GraphDatabase.driver("neo4j+s://cef6b56f.databases.neo4j.io", auth=("cef6b56f", "W6Dp8KwQSvXzQvITOeOm9NZXan5mwygBWObhY-WWb3w"))
neo4j_driver.verify_connectivity()

### Seeding

In [ ]:
reset_demo_graph(neo4j_driver)
ingest_demo_graph(neo4j_driver, neo4j_coverage_rows, neo4j_overlap_rows)
neo4j_driver.close()
'Aggregated graph created.'

### Reconnect To Aura From Scratch
Open a fresh connection after seeding so the live query cells start from a clean client.



In [39]:
manual_terms = ['gray rot', 'botrytis cinerea', 'clarification', 'filtration']
coverage_records = neo4j_driver.execute_query(
    DATASTORE_COVERAGE_CYPHER,
    routing_=RoutingControl.READ,
    demo_id=DEMO_ID,
    terms=[normalize_text(term) for term in manual_terms],
).records
pd.DataFrame(record.data() for record in coverage_records)



,datastore,concept_type,matched_term,coverage_count
0,wine_diseases,Disease,Botrytis cinerea,5
1,wine_making,Disease,gray rot,5
2,wine_diseases,Disease,gray rot,4
3,wine_making,Disease,Botrytis cinerea,4
4,wine_diseases,Keyword,gray rot,3
5,wine_making,Keyword,clarification,3
6,wine_making,Process,clarification,3
7,vine_diseases,Disease,Botrytis cinerea,2
8,vine_diseases,Disease,gray rot,2
9,vine_diseases,Keyword,Botrytis cinerea,2


In [40]:
overlap_records = neo4j_driver.execute_query(
    DATASTORE_OVERLAP_CYPHER,
    routing_=RoutingControl.READ,
    demo_id=DEMO_ID,
).records
pd.DataFrame(record.data() for record in overlap_records)



,left_datastore,right_datastore,score,shared_types,shared_terms
0,vine_diseases,wine_diseases,19,"[Disease, Keyword, QualityEffect, Region, Vari...","[Botrytis bunch rot, Botrytis cinerea, gray ro..."
1,wine_diseases,vine_diseases,19,"[Disease, Keyword, QualityEffect, Region, Vari...","[Botrytis bunch rot, Botrytis cinerea, gray ro..."
2,wine_diseases,wine_making,14,"[Disease, Keyword, Region, Variety]","[Botrytis cinerea, gray rot, noble rot, botryt..."
3,wine_making,wine_diseases,14,"[Disease, Keyword, Region, Variety]","[Botrytis cinerea, gray rot, noble rot, botryt..."
4,vine_diseases,wine_making,10,"[Disease, Region, Variety]","[Botrytis cinerea, gray rot, noble rot, Centra..."
5,wine_making,vine_diseases,10,"[Disease, Region, Variety]","[Botrytis cinerea, gray rot, noble rot, Centra..."


## LangGraph Runtime
Runtime graph: `triage_question -> route_with_neo4j -> retrieve_from_selected_datastores -> synthesize_answer`
No `RuntimeContext` is used here; the notebook globals act as the fixed dependencies.



In [48]:
class State(TypedDict, total=False):
    question: str
    triage: dict[str, Any]
    matched_graph_terms: list[str]
    selected_datastores: list[str]
    tool_requests: list[dict[str, Any]]
    retrieval_results: list[dict[str, Any]]
    final_answer: str


async def triage_question_node(state: State) -> dict[str, Any]:
    result = await triage_agent.run(f"User question:\n{state['question']}")
    triage = result.output
    print(f"Triage: ${triage.model_dump(mode='json')}")
    return {'triage': triage.model_dump(mode='json')}


def route_with_neo4j_node(state: State) -> dict[str, Any]:
    triage = QuestionTriage.model_validate(state['triage'])
    routed = route_datastores_with_neo4j(triage)
    print(f"matched_graph_terms: ${clean_list(term for row in routed for term in row['matched_terms'])}"),
    print(f"selected_datastores': ${[row['datastore'] for row in routed]}")

    return {
        'matched_graph_terms': clean_list(term for row in routed for term in row['matched_terms']),
        'selected_datastores': [row['datastore'] for row in routed],
        'tool_requests': routed,
    }


async def retrieve_from_selected_datastores_node(state: State) -> dict[str, Any]:
    triage = QuestionTriage.model_validate(state['triage'])
    tasks = []
    for datastore in state['selected_datastores']:
        if datastore == 'wine_making':
            tasks.append(qdrant_tool(state['question'], triage))
        else:
            tasks.append(sqlite_tool(state['question'], triage, datastore))
    retrieval_results = await asyncio.gather(*tasks)
    return {'retrieval_results': retrieval_results}


async def synthesize_answer_node(state: State) -> dict[str, Any]:
    retrieval_text = '\n\n'.join(
        f"[{result['datastore']}]\n{result['text']}"
        for result in state['retrieval_results']
        if result['text']
    )
    prompt = f"""
    User question:
    {state['question']}

    Retrieved text results:
    {retrieval_text}
    """.strip()
    final_answer = (await final_answer_agent.run(prompt)).output
    return {'final_answer': final_answer.answer}


workflow_builder = StateGraph(State)
workflow_builder.add_node('triage_question', triage_question_node)
workflow_builder.add_node('route_with_neo4j', route_with_neo4j_node)
workflow_builder.add_node('retrieve_from_selected_datastores', retrieve_from_selected_datastores_node)
workflow_builder.add_node('synthesize_answer', synthesize_answer_node)
workflow_builder.add_edge(START, 'triage_question')
workflow_builder.add_edge('triage_question', 'route_with_neo4j')
workflow_builder.add_edge('route_with_neo4j', 'retrieve_from_selected_datastores')
workflow_builder.add_edge('retrieve_from_selected_datastores', 'synthesize_answer')
workflow_builder.add_edge('synthesize_answer', END)
wine_demo_workflow = workflow_builder.compile()



## Live Questions
Each question has its own section so the exact wording stays visible during the demo.



## Demo Question 1
Goal: show a single-store route into the vineyard disease datastore.

Question:
"Which cultivars look safest for central Poland if fungal diseases are the main constraint?"



In [61]:
question_1 = 'Which cultivars look safest for central Poland if fungal diseases are the main constraint?'
question_1_result = await wine_demo_workflow.ainvoke({'question': question_1})
question_1_titles = []
for item in question_1_result['retrieval_results']:
    top_result = item.get('top_result')
    if isinstance(top_result, dict) and top_result:
        question_1_titles.append(top_result['title'])
    elif top_result is not None and getattr(top_result, 'payload', None):
        question_1_titles.append(top_result.payload.get('title'))
    else:
        question_1_titles.append(None)

{
    'question': question_1,
    'selected_datastores': question_1_result['selected_datastores'],
    'matched_graph_terms': question_1_result['matched_graph_terms'],
    'retrieval_titles': question_1_titles,
    'final_answer': question_1_result['final_answer'],
}



Triage: ${'regions': ['central Poland', 'Poland'], 'varieties': [], 'diseases': ['fungal diseases'], 'winemaking_steps': [], 'quality_effects': ['disease resistance'], 'keywords': ['central Poland cultivar choice', 'fungal disease pressure', 'disease-resistant cultivars', 'cold-climate viticulture'], 'requested_outcome': 'process_advice'}
matched_graph_terms: $['Central Poland', 'Poland', 'fungal diseases']
selected_datastores': $['vine_diseases']


{'question': 'Which cultivars look safest for central Poland if fungal diseases are the main constraint?',
 'selected_datastores': ['vine_diseases'],
 'matched_graph_terms': ['Central Poland', 'Poland', 'fungal diseases'],
 'retrieval_titles': ['Central Poland brief: why processing cultivars need both fungal tolerance and winter hardiness'],
 'final_answer': 'For central Poland, the safest cultivars appear to be interspecific hybrids, because the retrieved text says they showed less susceptibility to fungal diseases than classic Vitis vinifera cultivars. The same source also stresses winter hardiness, so the best-supported recommendation is hybrids used for processing rather than specific named cultivars. What is missing here is the list of individual cultivar names and their disease rankings.'}

## Demo Question 2
Goal: show a single-store route into the wine quality datastore.

Question:
"What signs tell a French winery that gray rot is already hurting wine quality?"



In [62]:
question_2 = 'What signs tell a French winery that gray rot is already hurting wine quality?'
question_2_result = await wine_demo_workflow.ainvoke({'question': question_2})
question_2_titles = []
for item in question_2_result['retrieval_results']:
    top_result = item.get('top_result')
    if isinstance(top_result, dict) and top_result:
        question_2_titles.append(top_result['title'])
    elif top_result is not None and getattr(top_result, 'payload', None):
        question_2_titles.append(top_result.payload.get('title'))
    else:
        question_2_titles.append(None)

{
    'question': question_2,
    'selected_datastores': question_2_result['selected_datastores'],
    'matched_graph_terms': question_2_result['matched_graph_terms'],
    'retrieval_titles': question_2_titles,
    'final_answer': question_2_result['final_answer'],
}



Triage: ${'regions': ['France'], 'varieties': [], 'diseases': ['gray rot'], 'winemaking_steps': [], 'quality_effects': ['wine quality deterioration', 'quality markers'], 'keywords': ['France', 'gray rot', 'wine quality signs', 'quality markers'], 'requested_outcome': 'quality_interpretation'}
matched_graph_terms: $['France', 'gray rot']
selected_datastores': $['wine_diseases']


{'question': 'What signs tell a French winery that gray rot is already hurting wine quality?',
 'selected_datastores': ['wine_diseases'],
 'matched_graph_terms': ['France', 'gray rot'],
 'retrieval_titles': ['Bordeaux brief: how little gray-rot fruit it can take before sensory quality starts to slip'],
 'final_answer': '{"answer": "In Bordeaux-focused INRAE results, gray rot is already hurting wine quality when Botrytis-affected grapes reach roughly 5%, because sensory quality starts to decline from that point. The main signs are mushroom or earthy notes, along with phenolic change and oxidation-linked damage in must and wine."}'}

## Demo Question 3
Goal: show a two-store fan-out that combines disease impact with cellar handling.

Question:
"Gray rot affected Pinot Noir grapes in France. How can it change wine quality, and what cellar step helps reduce clarification or filtration problems?"



In [63]:
parts = await wine_demo_workflow.ainvoke(
    {"question": question_3},
    version="v2",
    stream_mode=["updates"],
)

final_answer = None
for part in parts:
    if part["type"] != "updates":
        continue
    node_update = part["data"].get("synthesize_answer")
    if node_update and "final_answer" in node_update:
        final_answer = node_update["final_answer"]

print(final_answer)

Triage: ${'regions': ['France'], 'varieties': ['Pinot Noir'], 'diseases': ['gray rot'], 'winemaking_steps': ['clarification', 'filtration'], 'quality_effects': ['wine quality change', 'clarification problems', 'filtration problems'], 'keywords': ['gray rot Pinot Noir France', 'wine quality impact', 'cellar step to reduce filtration problems'], 'requested_outcome': 'mixed'}
matched_graph_terms: $['France', 'Pinot Noir', 'gray rot', 'clarification', 'filtration', 'clarification problems']
selected_datastores': $['wine_making', 'wine_diseases']
Gray rot here is Botrytis, and the French brief says wine sensory quality can decline from about 5% infected grapes onward, with phenolic change, oxidation-linked damage, and mushroom/earthy taints. For the cellar response, the wine-making brief states that treatment with beta-glucanases is specifically used to reduce clarification and filtration problems caused by beta-glucans from Botrytis-affected grapes.
